# Tests: `fasterai.regularize.regularize_callback` (source `nbs/regularize/regularize_callback.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.core.criteria import large_final
from fasterai.regularize.regularize_callback import *

In [ ]:
from fastcore.test import *

# Single criteria + granularity
cb = RegularizeCallback(criteria=large_final, granularity='filter', weight=1e-4)
test_eq(cb.weight, 1e-4)
test_eq(cb.current_weight, 1e-4)
test_eq(len(cb.criteria), 1)
test_eq(len(cb.granularity), 1)

# List of criteria/granularities
cb_m = RegularizeCallback(
    criteria=[large_final, large_final],
    granularity=['filter', 'weight']
)
test_eq(len(cb_m.criteria), 2)
test_eq(len(cb_m.granularity), 2)

# Default layer_types is Conv2d (listified)
test_eq(len(cb.layer_types), 1)
assert nn.Conv2d in cb.layer_types

# Schedule is None by default
test_eq(cb.schedule, None)

In [ ]:
#| slow
# Training with RegularizeCallback — verify it runs without error
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders
from fastai.learner import Learner

_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)
)

_X = torch.randn(64, 3, 8, 8)
_y = torch.randint(0, 10, (64,))
_dls = DataLoaders.from_dsets(
    TensorDataset(_X[:48], _y[:48]),
    TensorDataset(_X[48:], _y[48:]),
    bs=16, device='cpu'
)

_cb = RegularizeCallback(criteria=large_final, granularity='filter', weight=1e-4)
_learn = Learner(_dls, _model, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
_learn.fit(2)  # verify it runs end-to-end without error